Dawood I have checked your `test_pipeline.py` file. I want to add something about the approach...

I have change the name of my directory from scripts to services and you can find my modified `pipeline.py` there. You can see that I added the ids for spines because it is necessary for frontend and I declare the datatypes of variables as well. It is a good programming practice and required for FastAPI to check the validity of data.


I also have written the `text_cleaning.py` to clean the ocr results, then I saw yours and I just decided to discard it. I wrote a new code for text cleaning, you can find it in `text_testing.ipynb` file. And yeah!! I would suggest you to make another file for matching... where you can put your `text_cleaning` and `matching_title ` functions, after that you can delete your `test_pipeline.ipnb`. Make it compatible with my `pipeline.py`.


One more thing about relative path, I commented the relative path in `test_pipeline.py` for `books_cleaned.cvs` and it worked.



You can also leave your message below this... to keep the chat here. 😎

# SpinePipeline — Development Changes Summary

This document tracks every meaningful change made to `SpinePipeline` during development, in the order they happened, along with why each change was needed.

---

## 1. Initial pipeline (starting point)

The original `SpinePipeline` class covered the computer-vision side only:
- `detect_spines` — YOLO-based oriented bounding box detection of book spines
- `crop_spines` — perspective-corrects each detected spine into a straight rectangular crop
- `preprocess` — grayscale + CLAHE contrast enhancement + resizing before OCR
- `run_ocr` — runs PaddleOCR on each preprocessed crop
- `results` — orchestrates detection → cropping → OCR

## 2. Fixed a real bug in `run_ocr`

**Problem:** the method computed `filtered_texts`/`filtered_scores` (only keeping OCR results above `score_threshold`), but then appended the **unfiltered** `res['rec_texts']` to the output instead of the filtered versions.

**Fix:** append `filtered_texts`/`filtered_scores` instead, so low-confidence OCR fragments are actually excluded from the pipeline's output, as originally intended.

## 3. Added `extract_query_strings`

**Purpose:** PaddleOCR often splits a single spine's text into multiple fragments (e.g. title on one line, author on another). This method joins all fragments for one spine into a single string, ready for fuzzy matching.

## 4. Added Stage 1 — local catalog fuzzy matching (`match_books`)

**Approach:** used `rapidfuzz` with `token_set_ratio` to match each OCR query string against a combined `title + authors` field (`match_key`) built from the local catalog CSV.

**Why `rapidfuzz` / `token_set_ratio`:** OCR spine text is noisy (partial reads, word-order issues, typos) — token-based fuzzy matching tolerates this better than exact string matching or phonetic matching (which solves a different kind of noise: misspellings, not OCR character-confusion errors).

**Bug fixed along the way:** `isbn13` was being loaded by pandas as `int64`, which didn't match the string ids used in Chroma. Fixed by explicitly casting `catalog_df['isbn13']` to string in `__init__`.

## 5. Added Stage 2 — Chroma-based recommendations (`get_recommendations`)

**Approach:** given a matched book's `isbn13`, retrieve its already-stored embedding from Chroma and query for the most similar other books in the catalog (excluding the book itself).

**Key design decision:** book *identification* (Stage 1, fuzzy string matching) and book *recommendation* (Stage 2, semantic embedding similarity) are fundamentally different problems and were kept as separate stages rather than combined.

## 6. Fixed a model-naming collision bug

**Problem:** `self.model` was used for the YOLO detection model, but `results()` also called `self.model.encode(...)`, expecting a sentence-transformer — these were two different objects sharing one attribute name, causing an `AttributeError`.

**Fix:** added a separate `self.embedding_model` attribute specifically for the `SentenceTransformer` instance.

## 7. Made `results()` return richer, structured output

**Before:** `results()` returned a bare list of matched isbn13 strings.

**After:** returns a dictionary with three keys:
- `ocr_results` — raw PaddleOCR output per spine
- `query_strings` — joined title+author strings per spine
- `matches` — list of dicts, each with `ocr_text`, `matched_title`, `matched_authors`, `isbn13`, `score`, `source`, and `recommendations`

This gives full visibility into every pipeline stage for debugging, not just the final result.

## 8. Explored switching to a larger (60k-row) dataset — later abandoned

Attempted to replace the local catalog with a larger Kaggle dataset. This required:
- Swapping `isbn13` → `ID` throughout the pipeline
- Batching Chroma inserts (hit a hard `max batch size` limit of 5,461 per call)

**Outcome:** the new dataset was traced to a used-bookstore inventory source and lacked mainstream bestsellers entirely. This was reverted — the pipeline went back to using the original curated dataset. (See separate summary for full details on this detour.)

## 9. Added the Google Books API fallback (`fetch_from_google_books`)

**Purpose:** handle books that fail Stage 1 local matching, without relying on an LLM (which risks hallucinating book metadata). Google Books API is a factual lookup, not a generative one.

**Built-in robustness, added incrementally as real issues surfaced:**
- **Timeout + retry logic** — fixed an early problem where requests hung for 4-5 minutes with no timeout set. Added `timeout=5` and retry-with-backoff specifically for transient server errors (500/502/503/504).
- **Validation against the original query** — after Google returns a result, the returned title+authors are fuzzy-matched back against the original OCR query. If the score is too low, the result is rejected rather than trusted blindly (this caught a real case: querying just `"The Alchemist"` without an author returned an unrelated 1903 play instead of Paulo Coelho's novel).
- **Spin-off/derivative filtering (`is_likely_spinoff`)** — Google Books often ranks companion products (workbooks, journals, summaries) above the actual book. Added a keyword filter (`journal`, `workbook`, `summary`, etc.) to reject these.
- **Multi-candidate search (`maxResults=5`)** — instead of only checking Google's #1 result, the method now loops through up to 5 candidates and returns the first one that passes both the spin-off filter and the validation score — this directly fixed cases like "Zero to One," where the real book was only the 2nd or 3rd result.

## 10. Normalization fixes for fuzzy matching (`normalize_text`)

**Problem discovered:** OCR spine text is often ALL CAPS, while catalog/Google Books titles use normal casing. `token_set_ratio` compares tokens as exact strings, so `"ISLAM"` vs `"Islam"` scored far lower than it should have, despite being the same word.

**First attempt:** lowercase + strip **all whitespace**. This fixed the case problem but broke word-order tolerance (e.g. `"Friedrich Nietzsche Beyond Good and Evi"` vs `"Beyond Good and Evil Friedrich Nietzsche"` scored badly, since removing all spaces collapses each string into one giant token, defeating the point of token-based matching).

**Final fix:** lowercase + strip punctuation, but **keep spaces**. This preserves word-order invariance while still fixing the case-sensitivity issue.

## 11. Fixed OCR word-concatenation issues

**Problem:** OCR sometimes merges two words with no space (e.g. `PauloCoelho`, `ReclaimywaHeart`), especially with stylized/cursive fonts.

**Two angles tried:**
- Tuned PaddleOCR's `text_det_unclip_ratio` (controls how much detected text boxes are dilated/merged) — didn't meaningfully help, since visual inspection showed some of these were genuine tight cursive typography, not a detection bug.
- Added `split_concatenated_words` — a regex-based fix that inserts a space wherever a lowercase letter is immediately followed by an uppercase letter (`PauloCoelho` → `Paulo Coelho`). Applied in `extract_query_strings`, upstream of both Stage 1 and Stage 2 matching.

## 12. Tuned `match_score_cutoff`

Raised from `70` to `80` after noticing a false-positive match ("Big Magic" incorrectly matching "Stern Men" at a 70.8 score) once normalization made scoring more lenient overall. Verified against real test data that genuine matches still scored 90+ while this false positive disappeared.

## 13. Added diagnostic/debugging tools (not used in the live pipeline)

- **`debug_fetch`** — shows exactly why a Google Books query passed or failed (no results / HTTP error / spin-off rejected / validation score / missing description), since the main `fetch_from_google_books` fails silently by design.
- **`compare_scorers`** — runs the same OCR query strings through four different rapidfuzz scorers (`ratio`, `token_sort_ratio`, `token_set_ratio`, `partial_ratio`) side-by-side, to empirically compare which performs best on real data rather than guessing.
- **`display_results`** — pretty-prints full match details (title, source, OCR text, authors, isbn13, score, description preview, recommendations) instead of one-line summaries.

## 14. Fixed the embedding model reloading on every pipeline instantiation

**Problem:** `self.embedding_model = SentenceTransformer(...)` inside `__init__` reloaded the model from disk every time `SpinePipeline()` was instantiated, unlike `model` (YOLO) and `paddle_ocr`, which were loaded once at the module level and just imported.

**Fix:** moved the `SentenceTransformer` instantiation into `script/models.py` alongside the other two models, so it's loaded once and simply imported/reused across pipeline instances.

## 15. Added permanent caching of Google Books fallback results (`add_to_local_catalog`)

**Purpose:** avoid repeatedly calling the Google Books API for the same book across different users/sessions.

**What it does:**
- Generates a stable fallback id (hash of title+author) for books Google Books doesn't return an isbn13 for.
- Checks for duplicates two ways: exact isbn13 match, and a strict (score ≥ 90) fuzzy title+author match — since the same book could otherwise be cached twice under different ids if isbn13 is missing/inconsistent across different OCR reads.
- Adds the book's embedding to Chroma, and appends a full row (matching the catalog's actual schema) to both the in-memory `catalog_df` and the CSV file on disk.

## 16. Discovered and addressed a Chroma corruption incident

**Problem:** after some testing, `get_recommendations` started throwing an internal Chroma error (`"Error finding id"`) even for entries unrelated to the new caching feature — suggesting the collection's internal index had been corrupted, most likely from `collection.add()` being called with a duplicate id across separate sessions (since the hash-based fallback id would regenerate identically for the same book each time).

**Fix:** changed `add_to_local_catalog` to use `collection.upsert()` instead of `collection.add()` — `upsert` safely handles "insert if new, update if exists" without erroring or risking index corruption.

**Resolution:** rebuilt the Chroma database from scratch after clearing the corrupted one, and added `chroma_db/` to `.gitignore` to prevent large binary vector-database files from being committed to git (which had also caused separate git push issues).

---

## Current architecture (end state of this document)

```
Shelf photo
   ↓
YOLO spine detection → perspective-corrected crops
   ↓
CLAHE preprocessing → PaddleOCR text extraction (score-filtered)
   ↓
extract_query_strings (joins fragments, splits concatenated words)
   ↓
Stage 1: rapidfuzz local catalog matching (normalized, token_set_ratio, cutoff 80)
   ├─ Match found → Stage 2: Chroma similarity search → recommendations
   │
   └─ No match → Google Books API fallback
         ├─ Retry on transient errors, validate against OCR query,
         │   reject spin-off products, check up to 5 candidates
         ├─ Found → embed description locally → Chroma recommendations
         │            → cache into local catalog + Chroma (upsert)
         └─ Not found → silently dropped
```

The notebooks and dataset were lost while commiting and it about 5 Hour to resolve the issues with cmmiting and pushing. This took me 3-4 days because vectorizing 60000 books took 8-9 hours.

Chroma database is not comited because it took a lot of space and it was also causing problem while pushing

Using Google Books API and it works very good.

The model finds the books in image first in dataset based on fuzzymatch. If book is not in dataset then runs the google books api key to find book. If the boook is found in google books and isbn+tite don't match with any book n dataset, then add it to bith csv and chroma datbase 